# SimpleGPT - Training on Google Colab
Train a decoder-only GPT from scratch using the project's own modules.

## 1. Mount Google Drive (optional, for saving checkpoints)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone the repo

In [2]:
%cd /content
![ -d simple-chatbot/.git ] || git clone https://github.com/mrroy-dev/simple-chatbot.git
%cd /content/simple-chatbot

/content
/content/simple-chatbot


## 3. Install dependencies

In [4]:
!pip install torch pyyaml tqdm tensorboard matplotlib seaborn plotly


## 4. Check GPU

In [5]:
import torch
# IMPORTANT: If you get a CUDA error, do Runtime -> Restart runtime
try:
    torch.cuda.synchronize()
except Exception:
    print('CUDA error detected from previous run. Restart runtime and re-run.')
    raise

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

Using device: cuda
GPU: Tesla T4
Memory: 15.64 GB


## 5. Load or generate training data

In [6]:
import json, os, random, sys
os.makedirs('data', exist_ok=True)
random.seed(42)

train = None
val = None

# 1) Try pre-split files (upload these from your local machine)
train_path = 'data/pairs_train.json'
val_path = 'data/pairs_val.json'
if os.path.exists(train_path) and os.path.exists(val_path):
    with open(train_path) as f:
        train = json.load(f)
    with open(val_path) as f:
        val = json.load(f)
    print(f'Loaded pre-split data: {len(train)} train + {len(val)} val')

# 2) Fall back to single file (split automatically)
if train is None:
    data_path = 'data/pairs.json'
    if os.path.exists(data_path):
        print(f'Loading {data_path}...')
        with open(data_path) as f:
            all_pairs = json.load(f)
        print(f'Total entries: {len(all_pairs)}')
        random.shuffle(all_pairs)
        split = int(len(all_pairs) * 0.95)
        train = all_pairs[:split]
        val = all_pairs[split:]
        print(f'Split into {len(train)} train + {len(val)} val')

# 3) Last resort: synthetic data with warning
if train is None:
    print('WARNING: No data files found. Generating synthetic data.')
    print('Upload your own data/pairs_train.json and data/pairs_val.json for real training.')
    facts = {k: f'Definition of {k}.' for k in [
        'python', 'ml', 'dl', 'nn', 'transformers', 'gpt', 'pytorch',
        'gd', 'overfitting', 'attention', 'nlp', 'cnn', 'rnn', 'lstm',
        'gan', 'dropout', 'embedding', 'supervised', 'rl', 'backprop',
        'adam', 'softmax', 'relu', 'tensor', 'autograd',
    ]}
    questions = ['What is {t}?', 'Explain {t}', 'Tell me about {t}']
    all_pairs = []
    seen = set()
    for _ in range(3000):
        t = random.choice(list(facts.keys()))
        q = random.choice(questions).format(t=t)
        key = (q, facts[t])
        if key not in seen:
            seen.add(key)
            all_pairs.append({'context': f'<bos> <user> {q}', 'response': facts[t]})
    random.shuffle(all_pairs)
    print(f'Generated {len(all_pairs)} unique synthetic pairs (no duplicates)')
    split = int(len(all_pairs) * 0.95)
    train = all_pairs[:split]
    val = all_pairs[split:]

# Save for later cells
with open('data/pairs_train_subset.json', 'w') as f:
    json.dump(train, f)
with open('data/pairs_val_subset.json', 'w') as f:
    json.dump(val, f)

# Quick overlap check
train_pairs = set((e['context'], e['response']) for e in train)
val_pairs = set((e['context'], e['response']) for e in val)
overlap = train_pairs & val_pairs
print(f'Unique train pairs: {len(train_pairs)}')
print(f'Unique val pairs:   {len(val_pairs)}')
print(f'Train/val overlap:  {len(overlap)} ({"OK" if len(overlap)==0 else "LEAKAGE!"})')
print(f'Using {len(train)} train + {len(val)} val pairs')


Upload your own data/pairs_train.json and data/pairs_val.json for real training.
Generated 75 unique synthetic pairs (no duplicates)
Unique train pairs: 71
Unique val pairs:   4
Train/val overlap:  0 (OK)
Using 71 train + 4 val pairs


## 6. Import project modules
Uses the project's own BPETokenizer (with byte_encoder fix), ModelConfig, SimpleGPT (GQA, SwiGLU, RoPE, RMSNorm), and ConversationDataset.

In [ ]:
import os
import sys
import time
import math
import glob
import random
import inspect
from pathlib import Path

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from simple_gpt import ModelConfig, SimpleGPT, BPETokenizer
from simple_gpt.datasets.dataset import ConversationDataset

print('Imports OK')


## 7. Build tokenizer on the training data

In [ ]:
# Collect all text for training the BPE tokenizer
all_texts = []
for item in train:
    all_texts.append(item['context'])
    all_texts.append(item['response'])

tok = BPETokenizer()
tok._build_vocab(all_texts, min_freq=2, max_vocab_size=8000)

def effective_vocab_size(tokenizer):
    ids = []
    if getattr(tokenizer, 'vocab', None):
        ids.extend(tokenizer.vocab.values())
    if getattr(tokenizer, 'inv_vocab', None):
        ids.extend(tokenizer.inv_vocab.keys())
    if getattr(tokenizer, 'merges', None):
        ids.extend(tokenizer.merges.values())
    return max(ids) + 1 if ids else len(tokenizer)

tokenizer_vocab_size = effective_vocab_size(tok)
max_encoded_id = 0
for text in all_texts:
    ids = tok.encode(text)
    if ids:
        max_encoded_id = max(max_encoded_id, max(ids))

if max_encoded_id >= tokenizer_vocab_size:
    print(f'Expanding model vocab_size from {tokenizer_vocab_size} to {max_encoded_id + 1} '
          f'because tokenizer emits id {max_encoded_id}')
    tokenizer_vocab_size = max_encoded_id + 1

print(f'Tokenizer: __len__={tokenizer_vocab_size}  entries={len(tok.vocab)}  max_encoded_id={max_encoded_id}')
print(f'Special IDs: {tok.special_ids}')
print(f'PAD={tok.pad_id()} BOS={tok.bos_id()} EOS={tok.eos_id()}')


## 8. Create model using project's ModelConfig

In [ ]:
config = ModelConfig(
    vocab_size=tokenizer_vocab_size,
    block_size=256,
    n_layer=6,
    n_head=8,
    n_kv_head=4,
    n_embd=384,
    dropout=0.1,
    embd_dropout=0.1,
    weight_tying=True,
)
print(f'Model params: {sum(p.numel() for p in SimpleGPT(config).parameters()):,}')

model = SimpleGPT(config).to(device)
print(f'Model on {device} (vocab_size={config.vocab_size})')

## 9. Create datasets and data loaders

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = min(2, os.cpu_count() or 0)
PIN_MEMORY = device == 'cuda'
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

train_ds = ConversationDataset(train, tok, seq_len=config.block_size)
val_ds = ConversationDataset(val, tok, seq_len=config.block_size)

max_dataset_id = 0
for x, y, _, _ in train_ds:
    max_dataset_id = max(max_dataset_id, x.max().item(), y.max().item())
if max_dataset_id >= config.vocab_size:
    raise ValueError(
        f'dataset emits id {max_dataset_id}, but model vocab_size is {config.vocab_size}. '
        'Re-run cells 7, 8, and 9 in order so the model is rebuilt with the expanded vocab_size.'
    )

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
if NUM_WORKERS > 0:
    loader_kwargs.update(
        persistent_workers=PERSISTENT_WORKERS,
        prefetch_factor=PREFETCH_FACTOR,
    )

train_loader = DataLoader(
    train_ds,
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)
val_loader = DataLoader(
    val_ds,
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

print(f'Train: {len(train_ds)} examples, {len(train_loader)} batches, max_dataset_id={max_dataset_id}')
print(f'Val: {len(val_ds)} examples, {len(val_loader)} batches')
print(
    'DataLoader: '
    f'num_workers={NUM_WORKERS}, pin_memory={PIN_MEMORY}, '
    f'persistent_workers={PERSISTENT_WORKERS}, prefetch_factor={PREFETCH_FACTOR}'
)


## 10. Training setup
AdamW with weight decay, warmup-cosine LR scheduler.

In [ ]:
LR = 3e-4
MIN_LR = 3e-5
MAX_STEPS = 20000
WARMUP_STEPS = min(1000, max(500, int(0.02 * MAX_STEPS)))
LOG_EVERY = 10
VAL_EVERY = 100
GRAD_ACCUM_STEPS = 1
GRAD_CLIP_NORM = 1.0
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# AdamW parameter groups: decay matrix-like weights, skip biases and norm scalars.
# This follows GPT-style practice used by nanoGPT/GPT-2 training code.
decay_params = []
no_decay_params = []
decay_names = []
no_decay_names = []
seen_param_ids = set()
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if id(param) in seen_param_ids:
        continue
    seen_param_ids.add(id(param))

    lower_name = name.lower()
    if param.ndim < 2 or lower_name.endswith('bias') or 'norm' in lower_name or 'ln' in lower_name:
        no_decay_params.append(param)
        no_decay_names.append(name)
    else:
        decay_params.append(param)
        decay_names.append(name)

all_grouped_ids = {id(p) for p in decay_params} | {id(p) for p in no_decay_params}
trainable_ids = {id(p) for p in model.parameters() if p.requires_grad}
assert all_grouped_ids == trainable_ids, 'Optimizer parameter groups do not cover every trainable parameter exactly once.'
assert not ({id(p) for p in decay_params} & {id(p) for p in no_decay_params}), 'Parameter appears in both decay and no_decay groups.'

optimizer_kwargs = dict(lr=LR, betas=(0.9, 0.95), eps=1e-8)
if device == 'cuda' and 'fused' in inspect.signature(torch.optim.AdamW).parameters:
    optimizer_kwargs['fused'] = True

optimizer = torch.optim.AdamW(
    [
        {'params': decay_params, 'weight_decay': 0.1},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ],
    **optimizer_kwargs,
)

def lr_multiplier(step):
    """LambdaLR expects a multiplier of each param group's base lr, not an absolute lr."""
    if step < WARMUP_STEPS:
        return float(step + 1) / float(max(1, WARMUP_STEPS))
    progress = min(1.0, float(step - WARMUP_STEPS) / float(max(1, MAX_STEPS - WARMUP_STEPS)))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    min_lr_ratio = MIN_LR / LR
    return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_multiplier)

amp_device_type = 'cuda' if device == 'cuda' else 'cpu'
use_amp = device == 'cuda'
amp_dtype = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float16
try:
    scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and amp_dtype == torch.float16))
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and amp_dtype == torch.float16))

print('Optimizer & scheduler ready')
print(f'LR={LR:.2e}, MIN_LR={MIN_LR:.2e}, warmup={WARMUP_STEPS}, max_steps={MAX_STEPS}')
print(f'AdamW groups: decay={len(decay_params)} tensors, no_decay={len(no_decay_params)} tensors, fused={optimizer_kwargs.get("fused", False)}')
print(f'AMP: enabled={use_amp}, dtype={amp_dtype}, GradScaler enabled={scaler.is_enabled()}')


## 11. Training loop

In [ ]:
from tqdm.notebook import tqdm

# Keep CUDA asynchronous for performance. Set this to True only while debugging a CUDA stack trace.
DEBUG_CUDA_LAUNCH_BLOCKING = False
if DEBUG_CUDA_LAUNCH_BLOCKING:
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    print('CUDA_LAUNCH_BLOCKING=1 set for debugging')

print('=== Dataset Validation ===')
vocab_size = effective_vocab_size(tok)
vocab_entries = len(tok.vocab)
print(f'Tokenizer: len(tok)={len(tok)}, effective_vocab_size={vocab_size}, entries={vocab_entries}')
print(f'Model vocab_size: {model.config.vocab_size}')
print(f'Special IDs: {tok.special_ids}')
print(f'Max inv_vocab key: {max(tok.inv_vocab.keys())}')
missing_ids = [k for k in range(vocab_size) if k not in tok.inv_vocab]
missing_suffix = '...' if len(missing_ids) > 30 else ''
print(f'Missing inv_vocab keys: {missing_ids[:30]}{missing_suffix}')
assert model.config.vocab_size >= vocab_size, (
    f'model vocab_size={model.config.vocab_size} < tokenizer effective_vocab_size={vocab_size}; '
    're-run cells 7 and 8'
)

# Test one forward pass before the long run. Backward is covered by the first training micro-step.
tx, ty, tm, _ = next(iter(DataLoader(train_ds, batch_size=2, shuffle=False, num_workers=0)))
tx, ty, tm = tx.to(device, non_blocking=PIN_MEMORY), ty.to(device, non_blocking=PIN_MEMORY), tm.to(device, non_blocking=PIN_MEMORY)
model.eval()
with torch.no_grad():
    with torch.amp.autocast(device_type=amp_device_type, dtype=amp_dtype, enabled=use_amp):
        _, tloss, _ = model(tx, targets=ty, loss_mask=tm)
print(f'Validation forward: loss={tloss.item():.4f}')
model.train()

writer = SummaryWriter(log_dir='runs/simplegpt')
global_step = 0
best_val_loss = float('inf')
train_losses = []
val_losses = []
last_loss = None


def perplexity_from_loss(loss_value):
    return math.exp(min(float(loss_value), 20.0))


def current_gpu_memory_gb():
    if device != 'cuda':
        return 0.0
    return torch.cuda.max_memory_allocated() / 1e9


def checkpoint_payload(step, loss_value=None):
    payload = {
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict() if scaler is not None else None,
        'model_config': config,
        'tokenizer': tok,
        'step': step,
        'best_val_loss': best_val_loss,
        'val_loss': best_val_loss,
        'last_train_loss': loss_value,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'rng_state': torch.get_rng_state(),
        'python_random_state': random.getstate(),
    }
    if device == 'cuda':
        payload['cuda_rng_state_all'] = torch.cuda.get_rng_state_all()
    return payload


def save_checkpoint(step, *, is_best=False, tag='latest', loss_value=None):
    payload = checkpoint_payload(step, loss_value=loss_value)
    latest_path = CHECKPOINT_DIR / 'checkpoint_latest.pt'
    torch.save(payload, latest_path)
    if tag != 'latest':
        torch.save(payload, CHECKPOINT_DIR / f'checkpoint_{tag}.pt')
    if is_best:
        torch.save(payload, CHECKPOINT_DIR / 'checkpoint_best.pt')
        torch.save(payload, 'checkpoint_best.pt')  # Backward-compatible path used by chat/export cells.
    return latest_path


def find_latest_checkpoint():
    latest_path = CHECKPOINT_DIR / 'checkpoint_latest.pt'
    if latest_path.exists():
        return latest_path
    candidates = sorted(CHECKPOINT_DIR.glob('checkpoint_step_*.pt'))
    return candidates[-1] if candidates else None


def load_checkpoint_if_available():
    global global_step, best_val_loss, train_losses, val_losses
    ckpt_path = find_latest_checkpoint()
    if ckpt_path is None:
        print('No checkpoint found; starting fresh.')
        return
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    if ckpt.get('optimizer_state'):
        optimizer.load_state_dict(ckpt['optimizer_state'])
    if ckpt.get('scheduler_state'):
        scheduler.load_state_dict(ckpt['scheduler_state'])
    if scaler is not None and ckpt.get('scaler_state'):
        scaler.load_state_dict(ckpt['scaler_state'])
    global_step = int(ckpt.get('step', 0))
    best_val_loss = float(ckpt.get('best_val_loss', ckpt.get('val_loss', float('inf'))))
    train_losses = list(ckpt.get('train_losses', []))
    val_losses = list(ckpt.get('val_losses', []))
    print(f'Resumed from {ckpt_path} at step {global_step} with best_val_loss={best_val_loss:.4f}')


@torch.no_grad()
def run_validation(max_batches=None):
    was_training = model.training
    model.eval()
    total_loss = 0.0
    total_target_tokens = 0
    val_batches = 0
    try:
        for vx, vy, vm, _ in val_loader:
            target_tokens = int(vm.sum().item())
            vx = vx.to(device, non_blocking=PIN_MEMORY)
            vy = vy.to(device, non_blocking=PIN_MEMORY)
            vm = vm.to(device, non_blocking=PIN_MEMORY)
            with torch.amp.autocast(device_type=amp_device_type, dtype=amp_dtype, enabled=use_amp):
                _, vloss, _ = model(vx, targets=vy, loss_mask=vm)
            total_loss += float(vloss.item()) * max(target_tokens, 1)
            total_target_tokens += max(target_tokens, 1)
            val_batches += 1
            if max_batches is not None and val_batches >= max_batches:
                break
    finally:
        if was_training:
            model.train()
    val_loss = total_loss / max(total_target_tokens, 1)
    return val_loss, perplexity_from_loss(val_loss)


load_checkpoint_if_available()
model.train()
optimizer.zero_grad(set_to_none=True)
if device == 'cuda':
    torch.cuda.reset_peak_memory_stats()

pbar = tqdm(total=MAX_STEPS, initial=global_step, desc='Training')
last_log_time = time.time()
log_tokens = 0
running_loss = 0.0
log_micro_batches = 0
accum_micro_batches = 0
last_grad_norm = float('nan')

print(f'Model vocab_size = {model.config.vocab_size}')
print(f'Tokenizer len = {len(tok)}')
print(f'Tokenizer vocab entries = {len(tok.vocab)}')
print(f'block_size = {config.block_size}')
print(f'gradient_accumulation_steps = {GRAD_ACCUM_STEPS}')

while global_step < MAX_STEPS:
    for batch in train_loader:
        if global_step >= MAX_STEPS:
            break

        x, y, m, _ = batch
        log_tokens += int(x.numel())

        assert x.min() >= 0, f'x.min()={x.min()} < 0'
        assert y.min() >= 0, f'y.min()={y.min()} < 0'
        assert x.max() < model.config.vocab_size, f'x.max()={x.max()} >= vocab_size={model.config.vocab_size}'
        assert y.max() < model.config.vocab_size, f'y.max()={y.max()} >= vocab_size={model.config.vocab_size}'
        assert x.shape[1] <= model.block_size, f'seq_len={x.shape[1]} > block_size={model.block_size}'

        if global_step == 0 and log_micro_batches == 0 and accum_micro_batches == 0:
            print(
                f'First batch: x.shape={x.shape}, x.min={x.min()}, x.max={x.max()}, '
                f'y.min={y.min()}, y.max={y.max()}, block_size={model.block_size}'
            )

        x = x.to(device, non_blocking=PIN_MEMORY)
        y = y.to(device, non_blocking=PIN_MEMORY)
        m = m.to(device, non_blocking=PIN_MEMORY)

        with torch.amp.autocast(device_type=amp_device_type, dtype=amp_dtype, enabled=use_amp):
            _, loss, _ = model(x, targets=y, loss_mask=m)
            loss_to_backward = loss / GRAD_ACCUM_STEPS

        if not torch.isfinite(loss.detach()):
            raise FloatingPointError(f'Non-finite training loss at step {global_step}: {loss.item()}')

        if scaler.is_enabled():
            scaler.scale(loss_to_backward).backward()
        else:
            loss_to_backward.backward()

        last_loss = float(loss.detach().item())
        running_loss += last_loss
        log_micro_batches += 1
        accum_micro_batches += 1

        if accum_micro_batches < GRAD_ACCUM_STEPS:
            continue

        if scaler.is_enabled():
            scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        last_grad_norm = float(grad_norm.detach().cpu()) if torch.is_tensor(grad_norm) else float(grad_norm)
        if not math.isfinite(last_grad_norm):
            raise FloatingPointError(f'Non-finite gradient norm at step {global_step}: {last_grad_norm}')

        did_optimizer_step = True
        if scaler.is_enabled():
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            did_optimizer_step = scaler.get_scale() >= old_scale
        else:
            optimizer.step()

        optimizer.zero_grad(set_to_none=True)

        if did_optimizer_step:
            scheduler.step()
            global_step += 1
            accum_micro_batches = 0
            pbar.update(1)
        else:
            accum_micro_batches = 0
            print(f'Skipped optimizer/scheduler step at global_step={global_step} due to AMP overflow.')
            continue

        if global_step % LOG_EVERY == 0:
            now = time.time()
            elapsed = max(now - last_log_time, 1e-9)
            train_loss = running_loss / max(log_micro_batches, 1)
            lr = scheduler.get_last_lr()[0]
            tokens_per_sec = log_tokens / elapsed
            gpu_mem_gb = current_gpu_memory_gb()
            train_ppl = perplexity_from_loss(train_loss)

            train_losses.append((global_step, train_loss))
            writer.add_scalar('loss/train', train_loss, global_step)
            writer.add_scalar('perplexity/train', train_ppl, global_step)
            writer.add_scalar('lr', lr, global_step)
            writer.add_scalar('grad_norm', last_grad_norm, global_step)
            writer.add_scalar('throughput/tokens_per_sec', tokens_per_sec, global_step)
            writer.add_scalar('system/gpu_memory_gb', gpu_mem_gb, global_step)

            pbar.set_postfix(
                loss=f'{train_loss:.4f}',
                ppl=f'{train_ppl:.2f}',
                lr=f'{lr:.2e}',
                grad=f'{last_grad_norm:.2f}',
                tok_s=f'{tokens_per_sec:.0f}',
                mem=f'{gpu_mem_gb:.2f}GB',
            )

            running_loss = 0.0
            log_micro_batches = 0
            log_tokens = 0
            last_log_time = now

        if global_step % VAL_EVERY == 0:
            val_loss, val_ppl = run_validation()
            val_losses.append((global_step, val_loss))
            writer.add_scalar('loss/validation', val_loss, global_step)
            writer.add_scalar('perplexity/validation', val_ppl, global_step)

            is_best = val_loss < best_val_loss
            if is_best:
                best_val_loss = val_loss
            save_checkpoint(
                global_step,
                is_best=is_best,
                tag=f'step_{global_step:06d}',
                loss_value=last_loss,
            )
            status = 'new best' if is_best else 'val'
            print(f'Step {global_step}: {status} loss={val_loss:.4f}, ppl={val_ppl:.2f}, best={best_val_loss:.4f}')

pbar.close()
writer.flush()
writer.close()

save_checkpoint(global_step, is_best=False, tag='final', loss_value=last_loss)
torch.save(checkpoint_payload(global_step, loss_value=last_loss), 'checkpoint_final.pt')
print(f'Done! Best val_loss: {best_val_loss:.4f}')
print('Final checkpoint saved to checkpoint_final.pt and checkpoints/checkpoint_final.pt')


## Dataset validation note
Dataset and one-batch forward validation now run at the start of the training-loop cell so shape/vocab issues fail before training begins.


## 12. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
if train_losses:
    steps, losses = zip(*train_losses)
    plt.plot(steps, losses)
plt.xlabel('Step')
plt.ylabel('Train Loss')
plt.title('Training Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
if val_losses:
    steps, losses = zip(*val_losses)
    plt.plot(steps, losses, 'r-o')
plt.xlabel('Step')
plt.ylabel('Val Loss')
plt.title('Validation Loss')
plt.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()

if train_losses:
    print(f'Final train loss: {train_losses[-1][1]:.4f}')
print(f'Best val loss: {best_val_loss:.4f}')

# Optional post-training analysis utilities.
# Seaborn is nice for static publication-quality curves; Plotly is useful for interactive hover/zoom.
try:
    import pandas as pd
    import seaborn as sns
    metrics_rows = [{'step': s, 'loss': l, 'split': 'train'} for s, l in train_losses]
    metrics_rows += [{'step': s, 'loss': l, 'split': 'validation'} for s, l in val_losses]
    metrics_df = pd.DataFrame(metrics_rows)
    if not metrics_df.empty:
        plt.figure(figsize=(10, 4))
        sns.lineplot(data=metrics_df, x='step', y='loss', hue='split')
        plt.grid(True)
        plt.title('Train vs Validation Loss')
        plt.show()
except Exception as exc:
    print(f'Seaborn plot skipped: {exc}')

try:
    import plotly.express as px
    if 'metrics_df' in globals() and not metrics_df.empty:
        fig = px.line(metrics_df, x='step', y='loss', color='split', title='Train vs Validation Loss')
        fig.show()
except Exception as exc:
    print(f'Plotly plot skipped: {exc}')


## 13. Load best checkpoint and chat

In [ ]:
# Load best checkpoint.
# This checkpoint includes ModelConfig and tokenizer objects, so PyTorch 2.6+
# needs weights_only=False. Only use this for checkpoints you created/trust.
best_path = 'checkpoint_best.pt' if os.path.exists('checkpoint_best.pt') else str(CHECKPOINT_DIR / 'checkpoint_best.pt')
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model = SimpleGPT(ckpt['model_config']).to(device)
model.load_state_dict(ckpt['model_state'])
tok = ckpt['tokenizer']
model.eval()
print(
    f'Loaded best checkpoint from step {ckpt["step"]} '
    f'(best_val_loss={ckpt.get("best_val_loss", ckpt.get("val_loss", float("nan"))):.4f})'
)


## 14. Chat function

In [ ]:
@torch.no_grad()
def generate(model, tok, prompt, max_new=100, temp=0.7, top_k=40, top_p=0.9, rep_penalty=1.2):
    input_ids = [tok.bos_id()] + tok.encode(prompt) + [tok.bot_id()]
    idx = torch.tensor([input_ids], dtype=torch.long, device=device)

    for _ in range(max_new):
        if idx.size(1) > model.block_size:
            idx = idx[:, -model.block_size:]
        logits, _, _ = model(idx)
        logits = logits[:, -1, :] / max(temp, 1e-5)

        if rep_penalty is not None:
            for tid in set(idx[0].tolist()):
                logits[0, tid] /= rep_penalty

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_mask = cum_probs > top_p
            sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
            sorted_mask[..., 0] = 0
            indices_to_remove = sorted_mask.scatter(1, sorted_indices, sorted_mask)
            logits[indices_to_remove] = float('-inf')

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)

        if next_id.item() == tok.eos_id():
            break

    return tok.decode(idx[0].tolist())


def chat():
    print('\n=== SimpleGPT Chat ===')
    print('Type "quit" to exit, "clear" to reset history\n')
    history = []
    while True:
        user = input('You: ').strip()
        if user.lower() == 'quit':
            break
        if user.lower() == 'clear':
            history = []
            print('History cleared')
            continue
        if not user:
            continue
        context = ' '.join(history[-4:])
        prompt = f'<bos> <user> {user}' if not context else f'<bos> <user> {context} <user> {user}'
        response = generate(model, tok, prompt, max_new=100)
        print(f'Bot: {response}')
        history.extend([f'<user> {user}', f'<bot> {response}'])


chat()

## 15. Export to Google Drive

In [ ]:
import shutil
shutil.copy('checkpoint_best.pt', '/content/drive/MyDrive/') if os.path.exists('checkpoint_best.pt') else None
shutil.copy('checkpoint_final.pt', '/content/drive/MyDrive/') if os.path.exists('checkpoint_final.pt') else None
print('Copied to Drive' if os.path.exists('/content/drive') else 'Drive not mounted, skip export')